In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install the required libraries
!pip install geopandas fiona earthengine-api

# Import libraries
import geopandas as gpd
import ee
import os

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 63.2 MB/s eta 0:00:00


In [ ]:
# Enable all fiona drivers
import fiona
fiona.drvsupport.supported_drivers['KML'] = 'r'

# Path to your uploaded KML file
kml_file = "/content/drive/MyDrive/MINE_SIH2025/Data_aquisition/SIH2025_map.kml"

# Read the KML file into a GeoDataFrame
try:
    gdf = gpd.read_file(kml_file, driver='KML')

    # Assign original KML names to a new column 'name'
    if 'Name' in gdf.columns:
        gdf['name'] = gdf['Name']
    else:
        gdf['name'] = [f"poly_{i}" for i in range(len(gdf))]  # fallback

    print(f"Successfully loaded {len(gdf)} polygons from the KML file.")

    # Print the first few geometries with names
    for i, row in gdf.head(3).iterrows():
        print(f"Polygon {i} name:", row['name'])
        print(f"Polygon type:", row['geometry'].__geo_interface__['type'])
        print(f"Coordinates sample:", row['geometry'].__geo_interface__['coordinates'][0][:5])  # first 5 coords
        print("------")

except Exception as e:
    print(f"Error loading KML file: {e}")
    # Optional fallback with libkml driver
    # gdf = gpd.read_file(kml_file, driver='libkml')


Successfully loaded 18 polygons from the KML file.
Polygon 0 name: Jharia Coal mines
Polygon type: Polygon
Coordinates sample: ((86.13879332616892, 23.7573909676831, 0.0), (86.196932370041, 23.75882112076463, 0.0), (86.28163005074245, 23.77052302968364, 0.0), (86.36426413915443, 23.75416189456515, 0.0), (86.38369041400227, 23.72046927007425, 0.0))
------
Polygon 1 name: Govindpur Coal
Polygon type: Polygon
Coordinates sample: ((85.83673288196783, 23.81124267610327, 0.0), (85.83561240645724, 23.78769424230682, 0.0), (85.87083607900307, 23.7707043443951, 0.0), (85.92682184768765, 23.76067615589711, 0.0), (85.98045101656868, 23.75567293952414, 0.0))
------
Polygon 2 name: coal 3
Polygon type: Polygon
Coordinates sample: ((83.77054050205497, 21.97933627542355, 0.0), (83.76092415836641, 21.96209844595133, 0.0), (83.7690091853807, 21.94478179627273, 0.0), (83.78027194947829, 21.93250244242224, 0.0), (83.79756653163908, 21.92408992144536, 0.0))
------


In [ ]:
# Authenticate to your Google account and initialize the GEE API
ee.Authenticate()
ee.Initialize(project='mining-detection') # Replace with your project ID

In [ ]:
# ==============================
# STEP 5: Cleaning function
# ==============================
def clean_coords(coords):
    """Remove Z values (altitude) from coordinate tuples"""
    return [(x, y) for x, y, *_ in coords]

def get_ee_coords(gpd_json):
    """Convert GeoJSON geometry to Earth Engine-compatible coordinates"""
    if gpd_json['type'] == 'Polygon':
        return [clean_coords(ring) for ring in gpd_json['coordinates']]
    elif gpd_json['type'] == 'MultiPolygon':
        ee_coords = []
        for poly in gpd_json['coordinates']:
            for ring in poly:
                ee_coords.append(clean_coords(ring))
        return ee_coords
    else:
        return None

# ==============================
# STEP 6: Sentinel-2 Export Loop
# ==============================
drive_folder_name = "MINE_SIH2025_GEE_Data"  # Folder will be created inside your Google Drive
start_date = '2023-01-01'
end_date = '2023-12-31'
cloud_cover_max = 10
bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

for index, row in gdf.iterrows():
    geom = row['geometry']
    gpd_json = geom.__geo_interface__

    ee_coords = get_ee_coords(gpd_json)
    if ee_coords is None:
        print(f"Skipping geometry {index}: unsupported type {gpd_json['type']}")
        continue

    try:
        ee_polygon = ee.Geometry.Polygon(ee_coords)
    except Exception as e:
        print(f"⚠️ Error creating geometry for polygon {index}: {e}")
        continue

    # Sentinel-2 collection
    sentinel_collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(ee_polygon)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_cover_max))
    )

    # Cloud-free median composite
    image = sentinel_collection.select(bands).median().clip(ee_polygon)

    # Unique file name
    file_name = f"mine_stack_{index}"

    # Export task
    try:
        task = ee.batch.Export.image.toDrive(
            image=image.toFloat(),
            description=file_name,
            folder=drive_folder_name,
            fileNamePrefix=file_name,
            region=ee_polygon.bounds().getInfo()['coordinates'],
            scale=10,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        task.start()
        print(f"✅ Started export for polygon {index}: {file_name}")
    except Exception as e:
        print(f"⚠️ Export failed for polygon {index}: {e}")

✅ Started export for polygon 0: mine_stack_0
✅ Started export for polygon 1: mine_stack_1
✅ Started export for polygon 2: mine_stack_2
✅ Started export for polygon 3: mine_stack_3
✅ Started export for polygon 4: mine_stack_4
✅ Started export for polygon 5: mine_stack_5
✅ Started export for polygon 6: mine_stack_6
✅ Started export for polygon 7: mine_stack_7
✅ Started export for polygon 8: mine_stack_8
✅ Started export for polygon 9: mine_stack_9
✅ Started export for polygon 10: mine_stack_10
✅ Started export for polygon 11: mine_stack_11
✅ Started export for polygon 12: mine_stack_12
✅ Started export for polygon 13: mine_stack_13
✅ Started export for polygon 14: mine_stack_14
✅ Started export for polygon 15: mine_stack_15
✅ Started export for polygon 16: mine_stack_16
✅ Started export for polygon 17: mine_stack_17


In [ ]:
import ee, geopandas as gpd
import random


# ===== PARAMETERS =====
recent_year = 2024
bands = ['B2','B3','B4','B8','B11','B12']
patch_size_m = 5120
max_exports = 5
patches_per_polygon = 5
export_folder = "GEE_exports"

# ===== LOAD POLYGONS =====
gdf = gpd.read_file("/content/drive/MyDrive/MINE_SIH2025/Data_aquisition/SIH2025_map.kml")
print(f"Loaded {len(gdf)} polygons from KML")

# Reproject to UTM for correct area calculation
gdf = gdf.to_crs("EPSG:32644")
gdf['area'] = gdf.geometry.area
gdf = gdf.sort_values('area', ascending=False)

# Reproject back to WGS84 for GEE
gdf = gdf.to_crs("EPSG:4326")

# ===== HELPER FUNCTIONS =====
def clean_coords(coords):
    return [(x, y) for x, y, *_ in coords]

def to_ee_geometry(geom):
    gpd_json = geom.__geo_interface__
    if gpd_json['type'] == 'Polygon':
        ee_coords = [clean_coords(ring) for ring in gpd_json['coordinates']]
    elif gpd_json['type'] == 'MultiPolygon':
        ee_coords = []
        for poly in gpd_json['coordinates']:
            for ring in poly:
                ee_coords.append(clean_coords(ring))
    return ee.Geometry.Polygon(ee_coords)

# ===== SENTINEL-2 COLLECTION =====
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterDate(f"{recent_year}-01-01", f"{recent_year}-12-31")
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
      .select(bands))

# ===== MAIN LOOP =====
export_count = 0
for idx, row in gdf.iterrows():
    if export_count >= max_exports:
        break

    ee_poly = to_ee_geometry(row.geometry)

    # Median composite for entire polygon
    col = s2.filterBounds(ee_poly)
    try:
        if col.size().getInfo() == 0:
            print(f"No images for polygon {idx} in {recent_year}")
            continue
    except:
        continue

    comp = col.median().clip(ee_poly)  # ✅ Geometry

    # Randomly shuffle seeds to increase spatial variation
    seeds = random.sample(range(1000, 10000), patches_per_polygon)

    for j in range(patches_per_polygon):
        if export_count >= max_exports:
            break

        # Sample a random point for this patch
        points = ee.FeatureCollection.randomPoints(
            region=ee_poly,
            points=1,
            seed=seeds[j]
        )

        # Convert to square region
        reg = ee.Geometry(points.first().geometry()).buffer(patch_size_m/2).bounds()

        # Start export
        task = ee.batch.Export.image.toDrive(
            image=comp,
            description=f"mine_patch_poly{idx}_{recent_year}_{j}",
            folder=export_folder,
            region=reg,
            scale=10,
            crs="EPSG:4326",
            fileFormat="GeoTIFF",
            maxPixels=1e13
        )
        task.start()
        export_count += 1
        print(f"✅ Export {export_count}/{max_exports}: poly {idx}, patch {j}")

print(f"✅ Total exports started: {export_count}")


Loaded 18 polygons from KML
✅ Export 1/5: poly 0, patch 0
✅ Export 2/5: poly 0, patch 1
✅ Export 3/5: poly 0, patch 2
✅ Export 4/5: poly 0, patch 3
✅ Export 5/5: poly 0, patch 4
✅ Total exports started: 5


In [ ]:
import ee, geopandas as gpd
import random

# ===== PARAMETERS =====
years = [2024, 2023, 2022]        # most recent first
bands = ['B2','B3','B4','B8','B11','B12']
patch_size_m = 10240              # 1024 px * 10m resolution
max_exports = 200                 # total GeoTIFFs
patches_per_polygon = 5
export_folder = "GEE_exports"

# ===== LOAD POLYGONS =====
gdf = gpd.read_file("/content/drive/MyDrive/MINE_SIH2025/Data_aquisition/SIH2025_map.kml")
print(f"Loaded {len(gdf)} polygons from KML")

# Reproject to UTM for accurate area calculation
gdf = gdf.to_crs("EPSG:32644")
gdf['area'] = gdf.geometry.area
gdf = gdf.sort_values('area', ascending=False)

# Reproject back to WGS84 for GEE
gdf = gdf.to_crs("EPSG:4326")

# ===== HELPER FUNCTIONS =====
def clean_coords(coords):
    return [(x, y) for x, y, *_ in coords]

def to_ee_geometry(geom):
    gpd_json = geom.__geo_interface__
    if gpd_json['type'] == 'Polygon':
        ee_coords = [clean_coords(ring) for ring in gpd_json['coordinates']]
    elif gpd_json['type'] == 'MultiPolygon':
        ee_coords = []
        for poly in gpd_json['coordinates']:
            for ring in poly:
                ee_coords.append(clean_coords(ring))
    return ee.Geometry.Polygon(ee_coords)

# ===== MAIN LOOP =====
export_count = 0

for year in years:
    # Sentinel-2 collection for the current year
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterDate(f"{year}-01-01", f"{year}-12-31")
          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
          .select(bands))

    for idx, row in gdf.iterrows():
        if export_count >= max_exports:
            break

        ee_poly = to_ee_geometry(row.geometry)
        poly_name = row.get('Name', f'poly_{idx}')
        col = s2.filterBounds(ee_poly)

        try:
            if col.size().getInfo() == 0:
                print(f"No images for polygon {idx} ({poly_name}) in {year}")
                continue
        except:
            continue

        comp = col.median()  # Median composite (do not clip)
        seeds = random.sample(range(1000, 10000), patches_per_polygon)

        # Try shrinking bounds, fallback to full bounds if empty
        shrunk_bounds = ee_poly.bounds().buffer(-patch_size_m / 2)
        bounds = ee.Algorithms.If(
            shrunk_bounds.area().gt(0),
            shrunk_bounds,
            ee_poly.bounds()
        )

        for j in range(patches_per_polygon):
            if export_count >= max_exports:
                break

            point = ee.FeatureCollection.randomPoints(
                region=bounds,
                points=1,
                seed=seeds[j]
            ).first().geometry()

            patch = point.buffer(patch_size_m / 2).bounds()

            # Start export with polygon name and year in filename
            task = ee.batch.Export.image.toDrive(
                image=comp,
                description=f"{poly_name}_{year}_{j}",
                folder=export_folder,
                region=patch,
                scale=10,
                crs="EPSG:4326",
                fileFormat="GeoTIFF",
                maxPixels=1e13
            )
            task.start()
            export_count += 1
            print(f"✅ Export {export_count}/{max_exports}: poly {idx} ({poly_name}), year {year}, patch {j}")

    if export_count >= max_exports:
        break

print(f"✅ Total exports started: {export_count}")


Loaded 18 polygons from KML
✅ Export 1/200: poly 0 (Jharia Coal mines), year 2024, patch 0
✅ Export 2/200: poly 0 (Jharia Coal mines), year 2024, patch 1
✅ Export 3/200: poly 0 (Jharia Coal mines), year 2024, patch 2
✅ Export 4/200: poly 0 (Jharia Coal mines), year 2024, patch 3
✅ Export 5/200: poly 0 (Jharia Coal mines), year 2024, patch 4
✅ Export 6/200: poly 1 (Govindpur Coal), year 2024, patch 0
✅ Export 7/200: poly 1 (Govindpur Coal), year 2024, patch 1
✅ Export 8/200: poly 1 (Govindpur Coal), year 2024, patch 2
✅ Export 9/200: poly 1 (Govindpur Coal), year 2024, patch 3
✅ Export 10/200: poly 1 (Govindpur Coal), year 2024, patch 4
✅ Export 11/200: poly 8 (coal 9), year 2024, patch 0
✅ Export 12/200: poly 8 (coal 9), year 2024, patch 1
✅ Export 13/200: poly 8 (coal 9), year 2024, patch 2
✅ Export 14/200: poly 8 (coal 9), year 2024, patch 3
✅ Export 15/200: poly 8 (coal 9), year 2024, patch 4
✅ Export 16/200: poly 14 (coal 14), year 2024, patch 0
✅ Export 17/200: poly 14 (coal 14), 